# Make All Tables for Dissertation

Kendra Wyant  
March 27, 2026

In [ ]:
#| message: false
#| warning: false

suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(source("https://github.com/jjcurtin/lab_support/blob/main/format_path.R?raw=true"))
suppressPackageStartupMessages(library(tidyposterior))
library(kableExtra)


Attaching package: 'kableExtra'

The following object is masked from 'package:dplyr':

    group_rows

# Chapter 2: EMA

## Data and calculations

In [ ]:
disposition <- read_csv(file.path(path_processed_ema, "disposition.csv"), 
                        col_types = "ccDDcccccccccc")

screen <- read_csv(file.path(path_shared, "screen.csv"), 
                   col_types = cols()) |>
  filter(subid %in% subset(disposition, analysis == "yes")$subid) |> 
  mutate(across(dsm5_1:dsm5_11, ~ recode(., "No" = 0, "Yes" = 1))) |>  
  rowwise() |>  
  mutate(dsm5_total = sum(c(dsm5_1, dsm5_2, dsm5_3, dsm5_4, dsm5_5, dsm5_6, dsm5_7, 
                              dsm5_8, dsm5_9, dsm5_10, dsm5_11))) |>  
  ungroup()

lapses <- read_csv(file.path(path_shared, "lapses.csv"), col_types = cols()) |>
  filter(exclude == FALSE)

# Calcs to make df for table 1 (demographics and clinical characteristics)
n_total <- 151

dem_age <- screen |>
  summarise(mean = as.character(round(mean(dem_1, na.rm = TRUE), 1)),
            SD = as.character(round(sd(dem_1, na.rm = TRUE), 1)),
            min = as.character(min(dem_1, na.rm = TRUE)),
            max = as.character(max(dem_1, na.rm = TRUE))) |>
  mutate(var = "Age",
         n = as.numeric(""),
         perc = as.numeric("")) |>
  select(var, n, perc, everything()) 

dem_sex <-  screen |>
  select(var = dem_2) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / sum(n)) * 100) |> 
  add_row(var = "Sex", .before = 1)

dem_race <- screen |>
  select(var = dem_3) |>
  mutate(var = fct_relevel(factor(var,
                         c("American Indian/Alaska Native", "Asian", "Black/African American",
                           "White/Caucasian", "Other/Multiracial")))) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / sum(n)) * 100) |>
  add_row(var = "Race", .before = 1)
  
  
dem_ethnicity <- screen |>
  select(var = dem_4) |>
  mutate(var = case_when(var == "No, I am not of Hispanic, Latino, or Spanish origin" ~ "No",
                         TRUE ~ "Yes"),
         var = fct_relevel(factor(var, c("Yes", "No")))) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / sum(n)) * 100) |>
  add_row(var = "Hispanic, Latino, or Spanish origin", .before = 1)

dem_education <- screen |>
  select(var = dem_5) |>
  mutate(var = fct_relevel(factor(var,
                         c("Less than high school or GED degree", "High school or GED",
                           "Some college", "2-Year degree", "College degree", "Advanced degree")))) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / sum(n)) * 100) |>
  add_row(var = "Education", .before = 1)

dem_employment <- screen |>
  select(var = dem_6, dem_6_1) |>
  mutate(var = case_when(dem_6_1 == "Full-time" ~ "Employed full-time",
                         dem_6_1 == "Part-time" ~ "Employed part-time",
                         TRUE ~ var)) |>
  mutate(var = fct_relevel(factor(var,
                         c("Employed full-time", "Employed part-time", "Full-time student",
                           "Homemaker", "Disabled", "Retired", "Unemployed",
                           "Temporarily laid off, sick leave, or maternity leave",
                           "Other, not otherwise specified")))) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / sum(n)) * 100) |>
  add_row(var = "Employment", .before = 1)

dem_income <- screen |>
  summarise(mean = format(round(mean(dem_7, na.rm = TRUE), 0), big.mark = ","),
            SD = format(round(sd(dem_7, na.rm = TRUE), 0), big.mark = ","),
            min =format(round(min(dem_7, na.rm = TRUE), 0), big.mark = ","),
            max = format(round(max(dem_7, na.rm = TRUE), 0), scientific = FALSE, big.mark = ",")) |>
  mutate(var = "Personal Income",
        n = as.numeric(""),
        perc = as.numeric(""),
        mean = str_c("$", as.character(mean)),
        SD = str_c("$", as.character(SD)),
        min = str_c("$", as.character(min)),
        max = as.character(max)) |>
  select(var, n, perc, everything())

dem_marital <- screen |>
  select(var = dem_8) |>
  mutate(var = case_when(var == "Never Married" ~ "Never married",
                         TRUE ~ var)) |>
  mutate(var = fct_relevel(factor(var,
                         c("Never married", "Married", "Divorced", "Separated",
                           "Widowed")))) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / sum(n)) * 100) |> 
  add_row(var = "Marital Status", .before = 1)

dem_aud <- screen |>
  summarise(mean = as.character(round(mean(dsm5_total, na.rm = TRUE), 1)),
            SD = as.character(round(sd(dsm5_total, na.rm = TRUE), 1)),
            min = as.character(min(dsm5_total, na.rm = TRUE)),
            max = as.character(max(dsm5_total, na.rm = TRUE))) |>
  mutate(var = "DSM-5 AUD Symptom Count",
         n = as.numeric(""),
         perc = as.numeric("")) |>
  select(var, n, perc, everything()) 

lapses_per_subid <- screen |>
  select(subid) |>
  left_join(lapses |>
  janitor::tabyl(subid) |>
  select(-percent), by = "subid") |>
  mutate(n = if_else(is.na(n), 0, n),
         lapse = if_else(n > 0, "yes", "no"))

lapse_info <- lapses_per_subid |>
  group_by(lapse) |>
  rename(var = lapse) |>
  mutate(var = factor(var, levels = c("yes", "no"), labels = c("Yes", "No"))) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100,
         mean = NA_character_,
         SD = NA_character_,
         min = NA_character_,
         max = NA_character_) |>
  full_join(lapses_per_subid |>
  summarise(mean = as.character(round(mean(n), 1)),
            SD = as.character(round(sd(n), 1)),
            min = as.character(round(min(n), 1)),
            max = as.character(round(max(n), 1))) |>
  mutate(var = "Number of reported lapses"),
  by = c("var", "mean", "SD", "min", "max")) |> 
  add_row(var = "Reported 1 or More Lapse During Study Period", .before = 1)

table_dem <- dem_age |> 
  bind_rows(dem_sex) |> 
  bind_rows(dem_race) |> 
  bind_rows(dem_ethnicity) |> 
  bind_rows(dem_education) |> 
  bind_rows(dem_employment) |> 
  bind_rows(dem_income) |> 
  bind_rows(dem_marital) |> 
  bind_rows(dem_aud) |> 
  bind_rows(lapse_info) |> 
  mutate(range = str_c(min, "-", max),
         perc = round(perc, 1)) |> 
  select(-c(min, max)) |> 
  rename(N = n,
         `%` = perc,
         M = mean, 
         Range = range)

In [ ]:
# metrics
metrics_week <- read_csv(file.path(path_models_ema, "test_metrics_1week_0_v5_nested.csv"),
                         show_col_types = FALSE)
metrics_day <- read_csv(file.path(path_models_ema, "test_metrics_1day_0_v5_nested.csv"),
                        show_col_types = FALSE)
metrics_hour <- read_csv(file.path(path_models_ema, "test_metrics_1hour_0_v5_nested.csv"),
                         show_col_types = FALSE)

metrics <- metrics_week |> 
  mutate(model = "Week") |> 
  bind_rows(metrics_day |> 
              mutate(model = "Day")) |> 
  bind_rows(metrics_hour |> 
              mutate(model = "Hour")) |> 
  group_by(.metric, model) |> 
  summarize(median = median(.estimate), .groups = "drop") |> 
  pivot_wider(names_from = model, values_from = median) |> 
  select(.metric, Week, Day, Hour)

metrics <- metrics[c(4,5,6, 1, 3, 2),]

table_perf <- metrics |> 
 mutate(.metric = case_when(.metric == "roc_auc" ~ "auROC",
                            .metric == "sens" ~ "sensitivity",
                            .metric == "spec" ~ "specificity",
                            .metric == "bal_accuracy" ~ "balanced accuracy",
                            .metric == "ppv" ~ "positive predictive value",
                            .metric == "npv" ~ "negative predictive value")) |> 
 rename(Metric = .metric)

### Table 1: Demographic and Lapse Characteristics

In [ ]:
#| label: tbl-dem
#| tbl-cap: "Demographic and Lapse Characteristics"

table_dem |> 
  knitr::kable()

  -------------------------------------------------------------------------------------------
  var                                            N     \% M          SD         Range
  ------------------------------------------ ----- ------ ---------- ---------- -------------
  Age                                                     41         11.9       21-72

  Sex                                                                           

  Female                                        74   49.0                       

  Male                                          77   51.0                       

  Race                                                                          

  American Indian/Alaska Native                  3    2.0                       

  Asian                                          2    1.3                       

  Black/African American                         8    5.3                       

  White/Caucasian                              131   86.8                       

  Other/Multiracial                              7    4.6                       

  Hispanic, Latino, or Spanish origin                                           

  Yes                                            4    2.6                       

  No                                           147   97.4                       

  Education                                                                     

  Less than high school or GED degree            1    0.7                       

  High school or GED                            14    9.3                       

  Some college                                  41   27.2                       

  2-Year degree                                 14    9.3                       

  College degree                                58   38.4                       

  Advanced degree                               23   15.2                       

  Employment                                                                    

  Employed full-time                            72   47.7                       

  Employed part-time                            26   17.2                       

  Full-time student                              7    4.6                       

  Homemaker                                      1    0.7                       

  Disabled                                       7    4.6                       

  Retired                                        8    5.3                       

  Unemployed                                    18   11.9                       

  Temporarily laid off, sick leave, or           3    2.0                       
  maternity leave                                                               

  Other, not otherwise specified                 9    6.0                       

  Personal Income                                         \$34,298   \$31,807   \$0-200,000

  Marital Status                                                                

  Never married                                 67   44.4                       

  Married                                       32   21.2                       

  Divorced                                      45   29.8                       

  Separated                                      5    3.3                       

  Widowed                                        2    1.3                       

  DSM-5 AUD Symptom Count                                 8.9        1.9        4-11

  Reported 1 or More Lapse During Study                                         
  Period                                                                        

  Yes                                           84   55.6                       

  No                                            67   44.4                       

  Number of reported lapses                               6.8        12         0-75
  -------------------------------------------------------------------------------------------


### Table 2: Performance Metrics

In [ ]:
#| label: tbl-perf-ema
#| tbl-cap: "Performance Metrics for Full models by Prediction Window"
#| tbl-footnote: "Areas under the receiver operating characteristic curves (auROCs) summarize the model's sensitivity and specificity over all possible decision thresholds. Sensitivity, specificity, balanced accuracy, positive predictive value, and negative predictive value are performance metrics calculated at a single decision threshold for each model determined with Youden’s index. All metrics represent median values across 30 held-out test sets."

table_perf |> 
  knitr::kable()

  Metric                             Week         Day        Hour
  --------------------------- ----------- ----------- -----------
  auROC                         0.8906310   0.8989930   0.9293432
  sensitivity                   0.8232301   0.8275224   0.8636420
  specificity                   0.8192581   0.8454889   0.8813554
  balanced accuracy             0.8275871   0.8345976   0.8536644
  positive predictive value     0.6301088   0.2997223   0.0254053
  negative predictive value     0.9435759   0.9877148   0.9994361


# Chapter 3: Lag

Table 1

In [ ]:
ci_baseline <- read_csv(here::here(path_models_lag, "contrast_baseline.csv"), 
                        col_types = cols())

ci_lag <- read_csv(here::here(path_models_lag, "contrast_adjacent.csv"), 
                   col_types = cols())
ci_fairness_0 <- read_csv(here::here(path_models_lag, "pp_dem_contrast_all.csv"), 
                   col_types = cols()) |> 
  filter(lag == 0)

ci_fairness_336 <- read_csv(here::here(path_models_lag, "pp_dem_contrast_all.csv"), 
                   col_types = cols()) |> 
  filter(lag == 336)

table_ci <- ci_baseline |> 
  mutate(ci = str_c("[", round(lower, 3), ", ", round(upper, 3), "]"),
         median = as.character(round(median, 3)),
         probability = as.character(round(probability, 3))) |> 
  select(contrast, median, ci, probability) |> 
  add_row(contrast = "Baseline Contrasts", median = "", ci = "", probability = "") |> 
  mutate(contrast = factor(contrast,
                           levels = c("Baseline Contrasts",
                                      "0 vs. 24",
                                      "0 vs. 72",
                                      "0 vs. 168",
                                      "0 vs. 336"),
                           labels = c("Baseline Contrasts",
                                      "No lag vs. 1 day",
                                      "No lag vs. 3 days",
                                      "No lag vs. 1 week",
                                      "No lag vs. 2 weeks"))) |> 
  arrange(contrast) |> 
  rbind(ci_lag |> 
          mutate(ci = str_c("[", round(lower, 3), ", ", round(upper, 3), "]"),
                  median = as.character(round(median, 3)),
         probability = as.character(round(probability, 3))) |>
          select(contrast, median, ci, probability) |> 
          add_row(contrast = "Adjacent Contrasts", median = "", ci = "", 
                  probability = "") |> 
          mutate(contrast = factor(contrast,
                           levels = c("Adjacent Contrasts",
                                      "24 vs. 72",
                                      "72 vs. 168",
                                      "168 vs. 336"),
                           labels = c("Adjacent Contrasts",
                                      "1 day vs. 3 days",
                                      "3 days vs. 1 week",
                                      "1 week vs. 2 weeks"))) |> 
          arrange(contrast)) |> 
    rename(Contrast = contrast,
         Median = median,
         `Bayesian CI` = ci,
         Probability = probability)
 
  table_ci_fair <- ci_fairness_0 |> 
          mutate(ci = str_c("[", round(lower, 3), ", ", round(upper, 3), "]"),
                  median = as.character(round(median, 3)),
         probability = as.character(round(probability, 3))) |>
          select(contrast, median, ci, probability) |> 
          add_row(contrast = "Fairness Contrasts (No Lag)", median = "", ci = "", 
                  probability = "") |> 
          mutate(contrast = factor(contrast,
                           levels = c("Fairness Contrasts (No Lag)",
                                      "male vs female",
                                      "non-hispanic white vs not white",
                                      "above poverty vs below poverty"),
                           labels = c("Fairness Contrasts (No Lag)",
                                      "male vs. female",
                                      "non-Hispanic White vs. not White",
                                      "above poverty vs. below poverty"))) |> 
          arrange(contrast) |> 
    rbind(ci_fairness_336 |> 
          mutate(ci = str_c("[", round(lower, 3), ", ", round(upper, 3), "]"),
                  median = as.character(round(median, 3)),
         probability = as.character(round(probability, 3))) |>
          select(contrast, median, ci, probability) |> 
          add_row(contrast = "Fairness Contrasts (2-week Lag)", median = "", ci = "", 
                  probability = "") |> 
          mutate(contrast = factor(contrast,
                           levels = c("Fairness Contrasts (2-week Lag)",
                                      "male vs female",
                                      "non-hispanic white vs not white",
                                      "above poverty vs below poverty"),
                           labels = c("Fairness Contrasts (2-week Lag)",
                                      "male vs. female",
                                      "non-Hispanic White vs. not White",
                                      "above poverty vs. below poverty"))) |> 
          arrange(contrast)) |> 
  rename(Contrast = contrast,
         Median = median,
         `Bayesian CI` = ci,
         Probability = probability)

### Table 2: Model Comparisons

In [ ]:
#| label: tbl-model
#| tbl-cap: "Median difference in auROC, 95% Bayesian credible interval (CI), and posterior probability that that the auROC difference was smaller than 0 for all baseline and adjacent lag contrasts."

footnote_table_model <- "Median auROC differences less than 0 indicate the more lagged model, on average, performed worse than the more immediate model (e.g., 1-day lag - no lag, 3-day lag - 1-day lag). Bayesian CI represents the range of values where there is a 95% probability that the true auROC difference lies within that range. Probability indicates the posterior probability that this difference is smaller than 0 (i.e., the models are performing differently)."

table_ci |> 
  knitr::kable() |> 
  kable_classic() |> 
  kableExtra::column_spec(1, width = "25em") |> 
  kableExtra::group_rows(start_row = 2, end_row = 5) |> 
  kableExtra::group_rows(start_row = 7, end_row = 9) |> 
  kableExtra::row_spec(5, extra_css = "border-bottom: 1px solid") |> 
  kableExtra::add_footnote(label = footnote_table_model,
                           notation = "none",
                           escape = FALSE)

Contrast 
 Median 
 Bayesian CI 
 Probability 
 
 
 
 
 Baseline Contrasts 
 
 
 
 
 
 
 
 -0.042 
 [-0.048, -0.037] 
 1 
 
 
 
 -0.021 
 [-0.025, -0.017] 
 1 
 
 
 
 -0.063 
 [-0.069, -0.056] 
 1 
 
 
 
 -0.03 
 [-0.035, -0.026] 
 1 
 
 
 Adjacent Contrasts 
 
 
 
 
 
 
 
 -0.009 
 [-0.013, -0.005] 
 1 
 
 
 
 -0.012 
 [-0.017, -0.008] 
 1 
 
 
 
 -0.02 
 [-0.026, -0.015] 
 1 
 
 
 
 
 Median auROC differences less than 0 indicate the more lagged model, on average, performed worse than the more immediate model (e.g., 1-day lag - no lag, 3-day lag - 1-day lag). Bayesian CI represents the range of values where there is a 95% probability that the true auROC difference lies within that range. Probability indicates the posterior probability that this difference is smaller than 0 (i.e., the models are performing differently).

### Table 3: Fairness Comparisons

In [ ]:
#| label: tbl-fairness
#| tbl-cap: "Median difference in auROC, 95% Bayesian credible interval (CI), and posterior probability that that the auROC difference was smaller than 0 for fairness contrasts for the no lag and 2-week lagged models."

footnote_table_fair <- "Median auROC differences less than 0 indicate the model, on average, performed worse for the disadvantaged group (female, non-White and/or Hispanic, income below poverty line) compared to the advantaged group (male, non-Hispanic White, income above poverty line). Bayesian CI represents the range of values where there is a 95% probability that the true auROC difference lies within that range. Probability indicates the posterior probability that this difference is smaller than 0 (i.e., the models are performing differently for fairness subgroups)."

table_ci_fair |> 
  knitr::kable() |> 
  kable_classic() |> 
  kableExtra::column_spec(1, width = "25em") |> 
  kableExtra::group_rows(start_row = 2, end_row = 4) |> 
  kableExtra::group_rows(start_row = 6, end_row = 8) |> 
  kableExtra::row_spec(4, extra_css = "border-bottom: 1px solid")  |> 
  kableExtra::add_footnote(label = footnote_table_fair,
                           notation = "none",
                           escape = FALSE)

Contrast 
 Median 
 Bayesian CI 
 Probability 
 
 
 
 
 Fairness Contrasts (No Lag) 
 
 
 
 
 
 
 
 -0.043 
 [-0.059, -0.028] 
 1 
 
 
 
 -0.012 
 [-0.033, 0.007] 
 0.848 
 
 
 
 -0.131 
 [-0.222, -0.057] 
 0.999 
 
 
 Fairness Contrasts (2-week Lag) 
 
 
 
 
 
 
 
 -0.098 
 [-0.125, -0.073] 
 1 
 
 
 
 -0.039 
 [-0.073, -0.008] 
 0.98 
 
 
 
 -0.13 
 [-0.208, -0.058] 
 0.998 
 
 
 
 
 Median auROC differences less than 0 indicate the model, on average, performed worse for the disadvantaged group (female, non-White and/or Hispanic, income below poverty line) compared to the advantaged group (male, non-Hispanic White, income above poverty line). Bayesian CI represents the range of values where there is a 95% probability that the true auROC difference lies within that range. Probability indicates the posterior probability that this difference is smaller than 0 (i.e., the models are performing differently for fairness subgroups).

## Chapter 4

Demographics

In [ ]:
study_dates <- read_csv(file.path(path_risk2, "study_dates_gps.csv"), 
                        show_col_types = FALSE)

location <- read_csv(file.path(path_risk2, "features_fairness.csv"), 
                        show_col_types = FALSE) |>
  filter(subid %in% study_dates$subid) 

intake <- read_csv(file.path(path_risk2, "survey_intake.csv"), 
                   show_col_types = FALSE) |>
  filter(subid %in% study_dates$subid) 


# Calcs to make df for table 1 (demographics and clinical characteristics)
n_total <- nrow(study_dates)

dem_age <-  intake |>
  select(var = age) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |> 
  add_row(var = "Age", .before = 1)

dem_gender <-  intake |>
  select(var = gender) |>
  filter(var != "Prefer not to say") |> 
  mutate(var = factor(var, 
                      levels = c("Man",
                                 "Woman",
                                 "Non-binary",
                                 "Not listed above"))) |> 
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |> 
  add_row(var = "Gender", .before = 1)

dem_orientation <-  intake |>
  select(var = orientation) |>
  filter(!is.na(var)) |> 
  mutate(var = factor(var, levels = c("Straight, that is, not gay or lesbian",
                                      "Lesbian or gay",
                                      "Bisexual",
                                      "Not sure",
                                      "Not listed above"))) |> 
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |> 
  add_row(var = "Orientation", .before = 1)

dem_race <- intake |>
  select(starts_with("race_"), -race_other_text) |> 
  pivot_longer(1:7, names_to = "var") |> 
  mutate(var = case_match(var,
                           "race_ai_an" ~ "American Indian/Alaskan Native",
                           "race_asian" ~ "Asian",
                           "race_black" ~ "Black/African American",
                           "race_nat_hi" ~ "Native Hawaiian/Other Pacific Islander",
                           "race_white" ~ "White/Caucasian",
                           "race_hispanic" ~ "Hispanic, Latino, or Spanish origin",
                           "race_other" ~ "Not listed above"),
         var = factor(var,levels = c("American Indian/Alaskan Native",
                                     "Asian",
                                     "Black/African American",
                                     "Native Hawaiian/Other Pacific Islander",
                                     "White/Caucasian",
                                     "Hispanic, Latino, or Spanish origin",
                                     "Not listed above"))) |> 
  group_by(var) |>
  count(value) |> 
  ungroup() |> 
  filter(value == "yes") |> 
  select(-value) |> 
  mutate(perc = (n / n_total) * 100) |>
  add_row(var = "Race and Ethnicity (select all that apply)", .before = 1)

ℹ In argument: `var = case_match(...)`.
Caused by warning:
! `case_match()` was deprecated in dplyr 1.2.0.
ℹ Please use `recode_values()` instead.

In [ ]:
table_ci <- read_csv(here::here(path_models_risk2, "contrast_ablate_v10.csv"),
                    show_col_types = FALSE) 

### Table 1: Demographics and Clinical Characteristics

In [ ]:
#| label: tbl-risk2dem
#| tbl-cap: "Demographic and Clinical Characteristics"

footnote_table_dem_a <- "N = 296"


table_dem |> 
  knitr::kable() |> 
  kable_classic() |> 
  kableExtra::group_rows(start_row = 2, end_row = 7) |> 
  kableExtra::group_rows(start_row = 9, end_row = 12) |> 
  kableExtra::group_rows(start_row = 14, end_row = 18) |> 
  kableExtra::group_rows(start_row = 20, end_row = 26) |> 
  kableExtra::group_rows(start_row = 28, end_row = 29) |> 
  kableExtra::group_rows(start_row = 31, end_row = 36) |> 
  kableExtra::group_rows(start_row = 38, end_row = 43) |> 
  kableExtra::group_rows(start_row = 45, end_row = 52) |> 
  kableExtra::group_rows(start_row = 54, end_row = 57) |> 
  kableExtra::group_rows(start_row = 59, end_row = 60) |> 
  kableExtra::group_rows(start_row = 62, end_row = 63) |> 
  kableExtra::group_rows(start_row = 65, end_row = 66) |> 
  kableExtra::group_rows(start_row = 68, end_row = 71) |>
  kableExtra::group_rows(start_row = 73, end_row = 77) |>
  kableExtra::group_rows(start_row = 79, end_row = 83) |>
  kableExtra::footnote(general = c(footnote_table_dem_a), escape=FALSE) 

N 
 % 
 
 
 
 
 Age 
 
 
 
 
 
 22-25 
 7 
 2.4 
 
 
 26-35 
 105 
 35.5 
 
 
 36-45 
 107 
 36.1 
 
 
 46-55 
 60 
 20.3 
 
 
 56-65 
 13 
 4.4 
 
 
 Over 65 
 4 
 1.4 
 
 
 Gender 
 
 
 
 
 
 Man 
 158 
 53.4 
 
 
 Woman 
 131 
 44.3 
 
 
 Non-binary 
 4 
 1.4 
 
 
 Not listed above 
 2 
 0.7 
 
 
 Orientation 
 
 
 
 
 
 Straight, that is, not gay or lesbian 
 237 
 80.1 
 
 
 Lesbian or gay 
 17 
 5.7 
 
 
 Bisexual 
 35 
 11.8 
 
 
 Not sure 
 1 
 0.3 
 
 
 Not listed above 
 4 
 1.4 
 
 
 Race and Ethnicity (select all that apply) 
 
 
 
 
 
 American Indian/Alaskan Native 
 17 
 5.7 
 
 
 Asian 
 3 
 1.0 
 
 
 Black/African American 
 43 
 14.5 
 
 
 Native Hawaiian/Other Pacific Islander 
 3 
 1.0 
 
 
 White/Caucasian 
 235 
 79.4 
 
 
 Hispanic, Latino, or Spanish origin 
 26 
 8.8 
 
 
 Not listed above 
 5 
 1.7 
 
 
 Geographic Location 
 
 
 
 
 
 rural 
 57 
 19.3 
 
 
 urban 
 239 
 80.7 
 
 
 Education 
 
 
 
 
 
 8th grade or less 
 5 
 1.7 
 
 
 Some high school, but did not graduate 
 25 
 8.4 
 
 
 High school graduate or GED 
 92 
 31.1 
 
 
 Some college or 2-year degree 
 130 
 43.9 
 
 
 4-year college graduate 
 30 
 10.1 
 
 
 More than 4-year or advanced degree 
 12 
 4.1 
 
 
 Employment 
 
 
 
 
 
 Disabled, not able to work 
 34 
 11.5 
 
 
 Employed, working 1-39 hours per week 
 69 
 23.3 
 
 
 Employed, working 40 or more hours per week 
 54 
 18.2 
 
 
 Not employed, NOT looking for work 
 32 
 10.8 
 
 
 Not employed, looking for work 
 102 
 34.5 
 
 
 Retired 
 3 
 1.0 
 
 
 Household Income 
 
 
 
 
 
 Less than $25,000 
 164 
 55.4 
 
 
 $25,000 - $34,999 
 42 
 14.2 
 
 
 $35,000 - $49,999 
 38 
 12.8 
 
 
 $50,000 - $74,999 
 29 
 9.8 
 
 
 $75,000 - $99,999 
 13 
 4.4 
 
 
 $100,000 - $149,999 
 3 
 1.0 
 
 
 $150,000 - $199,999 
 4 
 1.4 
 
 
 $200,000 or more 
 1 
 0.3 
 
 
 Self-reported DSM-5 OUD Symptom Count 
 
 
 
 
 
 Under threshold (0-1) 
 5 
 1.7 
 
 
 Mild (2-3) 
 0 
 0.0 
 
 
 Moderate (4-5) 
 5 
 1.7 
 
 
 Severe (6+) 
 284 
 95.9 
 
 
 Past Month Opioid Use 
 
 
 
 
 
 No 
 201 
 67.9 
 
 
 Yes 
 93 
 31.4 
 
 
 Past Month Detox or Residential Treatment 
 
 
 
 
 
 No 
 189 
 63.9 
 
 
 Yes 
 106 
 35.8 
 
 
 Past Month Psychiatric Medication 
 
 
 
 
 
 No 
 151 
 51.0 
 
 
 Yes 
 144 
 48.6 
 
 
 Preferred Opioid 
 
 
 
 
 
 Fentanyl 
 50 
 16.9 
 
 
 Heroin 
 122 
 41.2 
 
 
 Prescription opioid not for opioid treatment 
 93 
 31.4 
 
 
 Medication for opioid treatment 
 28 
 9.5 
 
 
 Preferred Route of Administration 
 
 
 
 
 
 Injection 
 104 
 35.1 
 
 
 Oral 
 56 
 18.9 
 
 
 Smoke 
 48 
 16.2 
 
 
 Sniff or snort 
 83 
 28.0 
 
 
 Other 
 3 
 1.0 
 
 
 Lifetime History of Overdose 
 
 
 
 
 
 Never 
 119 
 40.2 
 
 
 1 time 
 41 
 13.9 
 
 
 2-3 times 
 69 
 23.3 
 
 
 4-5 times 
 26 
 8.8 
 
 
 More than 5 times 
 39 
 13.2 
 
 
 
 Note: 
 
 N = 296

### Table 2: Model Comparisons

In [ ]:
#| label: tbl-risk2model
#| tbl-cap: "Median difference in auROC, 95% Bayesian credible interval (CI), and posterior probability that that the auROC difference was larger than 0 for all ablated model contrasts."

footnote_table_model <- "Median auROC differences greater than 0 indicate the full model, on average, performed better than the ablated model (i.e., full vs. ablated daily survey, full vs. ablated geolocation, full vs. ablated daily survey and geolocation). Bayesian CI represents the range of values where there is a 95% probability that the true auROC difference lies within that range. Probability indicates the posterior probability that this difference is greater than 0 (i.e., the full model is performing better)."

table_ci |> 
  mutate(`Bayesian CI` = str_c("[", round(lower, 3), ", ", round(upper, 3), "]"),
         `Median auROC Difference` = as.character(round(median, 3)),
         Probability = sprintf("%.3f", probability)) |> 
  select(Contrast = contrast, `Median auROC Difference`, `Bayesian CI`, Probability) |> 
  knitr::kable() |> 
  kable_classic() |> 
  kableExtra::column_spec(1, width = "25em") |> 
  kableExtra::add_footnote(label = footnote_table_model,
                           notation = "none",
                           escape = FALSE) 

Contrast 
 Median auROC Difference 
 Bayesian CI 
 Probability 
 
 
 
 
 full vs. ablated daily survey 
 0.314 
 [0.29, 0.338] 
 1.000 
 
 
 full vs. ablated geolocation 
 -0.002 
 [-0.009, 0.005] 
 0.342 
 
 
 full vs. ablated daily survey and geolocation 
 0.389 
 [0.363, 0.414] 
 1.000 
 
 
 
 
 Median auROC differences greater than 0 indicate the full model, on average, performed better than the ablated model (i.e., full vs. ablated daily survey, full vs. ablated geolocation, full vs. ablated daily survey and geolocation). Bayesian CI represents the range of values where there is a 95% probability that the true auROC difference lies within that range. Probability indicates the posterior probability that this difference is greater than 0 (i.e., the full model is performing better).